In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import KFold
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from xgboost import XGBRegressor

print('Библиотеки загружены')

Библиотеки загружены


## 1. Загрузка данных

In [ ]:
# При работе в Google Colab раскомментируйте загрузку:
# !gdown https://drive.google.com/file/d/1ISE1fs4FJPy_SU3D9A5mftMxFKDjT9VA/view?usp=sharing --fuzzy
# !gdown https://drive.google.com/file/d/1We0hORnaZq8fzPLrMgx7aV0Ma9cVaRJj/view?usp=sharing --fuzzy

TRAIN_PATH = '/content/hackathon_income_train.csv'
TEST_PATH  = '/content/hackathon_income_test.csv'

train = pd.read_csv(TRAIN_PATH, sep=';', decimal=',', low_memory=False)
test  = pd.read_csv(TEST_PATH,  sep=';', decimal=',', low_memory=False)

print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Target stats:\n{train["target"].describe()}')

Train: (76786, 224), Test: (73214, 222)
Target stats:
count    7.678600e+04
mean     9.264824e+04
std      1.124090e+05
min      2.000000e+04
25%      3.970997e+04
50%      6.275413e+04
75%      1.002017e+05
max      1.500000e+06
Name: target, dtype: float64


## 2. Препроцессинг

In [ ]:
# Метрика
def wmae(y_true, y_pred, w):
    """Weighted Mean Absolute Error — целевая метрика соревнования."""
    return np.sum(w * np.abs(np.array(y_true) - np.array(y_pred))) / np.sum(w)

# Кастомная eval-функция для LightGBM (нужна для early stopping по WMAE)
def lgb_wmae_eval(y_pred, dataset):
    y_true = dataset.get_label()
    w      = dataset.get_weight()
    score  = wmae(y_true, y_pred, w)
    return 'wmae', score, False  # False = lower is better

# Кастомная eval-функция для XGBoost
def lgb_wmae_eval(y_pred, dataset):
    """Работает в любых версиях LightGBM"""
    if isinstance(dataset, np.ndarray):
        y_true = dataset
        w = np.ones_like(y_true)
    else:
        y_true = dataset.get_label()
        w = dataset.get_weight() if dataset.get_weight() is not None else np.ones_like(y_true)

    score = wmae(y_true, y_pred, w)
    return 'wmae', score, False

print('Метрика WMAE определена')

Метрика WMAE определена


In [ ]:
# Столбцы для удаления
# Служебные
SERVICE_COLS = ['id', 'dt', 'w', 'target']

# Константные в данных (только 1 уникальное значение)
CONSTANT_COLS = [
    'dp_ewb_dismissal_due_contract_violation_by_lb_cnt',
    'dp_ils_days_ip_share_5y',
    'period_last_act_ad',
]

DROP_COLS = SERVICE_COLS + CONSTANT_COLS

# Функция препроцессинга
def preprocess_raw(df):
    df = df.copy()

    object_cols = df.select_dtypes(include='object').columns.tolist()
    object_cols = [c for c in object_cols if c not in DROP_COLS]

    for col in object_cols:
        converted = pd.to_numeric(
            df[col].astype(str).str.replace(',', '.', regex=False).str.strip(),
            errors='coerce'
        )
        if converted.notna().mean() > 0.3:
            df[col] = converted
        else:
            pass

    return df


def add_features(df):
    """
    Добавляет новые признаки поверх существующих.
    Вызывается ПОСЛЕ заполнения пропусков.
    """
    # Квадрат возраста (нелинейность)
    if 'age' in df.columns:
        df['feat_age_sq'] = df['age'] ** 2

    # Тренд выплат (растут или падают)
    p6 = 'dp_payoutincomedata_payout_avg_6_month'
    p3 = 'dp_payoutincomedata_payout_avg_3_month'
    if p6 in df.columns and p3 in df.columns:
        df['feat_payout_trend'] = df[p6] - df[p3]
        df['feat_payout_ratio'] = df[p3] / (df[p6] + 1)

    # Динамика зарплаты по годам
    s1 = 'dp_ils_avg_salary_1y'
    s2 = 'dp_ils_avg_salary_2y'
    s3 = 'dp_ils_avg_salary_3y'
    if s2 in df.columns and s1 in df.columns:
        df['feat_salary_growth_1y'] = df[s2] - df[s1]
    if s3 in df.columns and s2 in df.columns:
        df['feat_salary_growth_2y'] = df[s3] - df[s2]

    # Чистый входящий поток
    cr = 'turn_cur_cr_avg_act_v2'
    db = 'turn_cur_db_avg_act_v2'
    if cr in df.columns and db in df.columns:
        df['feat_net_flow'] = df[db] - df[cr]
        df['feat_flow_ratio'] = df[cr] / (df[db] + 1)

    # Log-трансформация для сильно скошенных ключевых признаков
    log_cols = [
        'turn_cur_cr_avg_act_v2', 'turn_cur_db_avg_act_v2',
        'avg_6m_all', 'avg_3m_all',
        'dp_payoutincomedata_payout_avg_6_month',
        'first_salary_income', 'salary_6to12m_avg',
    ]
    for col in log_cols:
        if col in df.columns:
            df[f'feat_log_{col}'] = np.log1p(np.clip(df[col], 0, None))

    # Флаг есть ли данные о зарплате из ПФР/ФНС
    if 'first_salary_income' in df.columns:
        df['feat_has_salary_data'] = df['first_salary_income'].notna().astype(int)
    if p6 in df.columns:
        df['feat_has_payout_data'] = df[p6].notna().astype(int)

    return df

print('Функции препроцессинга определены')

Функции препроцессинга определены


In [ ]:
# Частотное кодирование категориальных
def fit_frequency_encoding(train_df):
    """Считает частотные карты по train."""
    cat_cols = train_df.select_dtypes(include='object').columns.tolist()
    cat_cols = [c for c in cat_cols if c not in DROP_COLS]
    freq_maps = {}
    for col in cat_cols:
        freq_maps[col] = train_df[col].value_counts(normalize=True)
    return freq_maps

def apply_frequency_encoding(df, freq_maps):
    """Применяет частотные карты. Незнакомые значения → 0."""
    df = df.copy()
    for col, freq_map in freq_maps.items():
        if col in df.columns:
            df[col] = df[col].map(freq_map).fillna(0)
    return df

# Target encoding по adminarea и city_smart_name
def fit_target_encoding(train_df, target, smoothing=50):
    """Target encoding с Bayesian smoothing для предотвращения переобучения."""
    global_mean = np.mean(target)
    te_maps = {}
    cat_cols_for_te = ['adminarea', 'city_smart_name']
    for col in cat_cols_for_te:
        if col not in train_df.columns:
            continue
        stats = train_df.groupby(col)['__target__'].agg(['mean', 'count'])
        smoothed = (stats['mean'] * stats['count'] + global_mean * smoothing) / (stats['count'] + smoothing)
        te_maps[col] = smoothed
    return te_maps, global_mean

def apply_target_encoding(df, te_maps, global_mean):
    df = df.copy()
    for col, te_map in te_maps.items():
        new_col = f'te_{col}'
        if col in df.columns:
            df[new_col] = df[col].map(te_map).fillna(global_mean)
    return df

print('Функции кодирования определены')

Функции кодирования определены


In [ ]:
# Предварительная сырая конвертация типов
train_raw = preprocess_raw(train)
test_raw  = preprocess_raw(test)

target_arr  = train['target'].values
weights_arr = train['w'].values
test_ids    = test['id']

print(f'После конвертации типов — train object cols: {train_raw.select_dtypes("object").shape[1]}')
print(f'После конвертации типов — test  object cols: {test_raw.select_dtypes("object").shape[1]}')

После конвертации типов — train object cols: 13
После конвертации типов — test  object cols: 13


## 3. OOF Кросс-валидация — LightGBM

In [ ]:
N_FOLDS   = 5
SEED      = 42

# Параметры LightGBM
LGB_PARAMS = dict(
    objective         = 'regression_l1',
    n_estimators      = 10000,
    learning_rate     = 0.02,
    num_leaves        = 127,
    max_depth         = -1,
    min_child_samples = 30,
    feature_fraction  = 0.7,
    bagging_fraction  = 0.8,
    bagging_freq      = 1,
    reg_alpha         = 0.1,
    reg_lambda        = 1.0,
    random_state      = SEED,
    n_jobs            = -1,
    verbose           = -1,
)

kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

oof_lgb  = np.zeros(len(train_raw))
test_lgb = np.zeros(len(test_raw))
fold_scores_lgb = []
lgb_best_iters  = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(train_raw)):
    print(f'\n── Fold {fold+1}/{N_FOLDS} ──')

    train_fold = train_raw.iloc[tr_idx].copy()
    valid_fold = train_raw.iloc[val_idx].copy()

    y_tr = target_arr[tr_idx]
    y_val = target_arr[val_idx]
    w_tr = weights_arr[tr_idx]
    w_val = weights_arr[val_idx]

    # 1) Частотное кодирование
    freq_maps = fit_frequency_encoding(train_fold)
    train_fold = apply_frequency_encoding(train_fold, freq_maps)
    valid_fold = apply_frequency_encoding(valid_fold, freq_maps)
    test_fold  = apply_frequency_encoding(test_raw.copy(), freq_maps)

    # 2) Target encoding
    train_fold['__target__'] = y_tr
    te_maps, global_mean = fit_target_encoding(train_fold, y_tr)
    train_fold = train_fold.drop(columns=['__target__'])
    train_fold = apply_target_encoding(train_fold, te_maps, global_mean)
    valid_fold = apply_target_encoding(valid_fold, te_maps, global_mean)
    test_fold  = apply_target_encoding(test_fold, te_maps, global_mean)

    # 3) Заполнение пропусков медианой по train-fold
    drop_present = [c for c in DROP_COLS if c in train_fold.columns]
    X_tr  = train_fold.drop(columns=drop_present)
    X_val = valid_fold.drop(columns=[c for c in DROP_COLS if c in valid_fold.columns])
    X_te  = test_fold.drop(columns=[c for c in DROP_COLS if c in test_fold.columns])

    medians = X_tr.median()
    X_tr  = X_tr.fillna(medians)
    X_val = X_val.fillna(medians)
    X_te  = X_te.fillna(medians)

    # 4) Общие колонки
    common_cols = [c for c in X_tr.columns if c in X_val.columns and c in X_te.columns]
    X_tr, X_val, X_te = X_tr[common_cols], X_val[common_cols], X_te[common_cols]

    # 5) Добавление новых признаков
    X_tr  = add_features(X_tr)
    X_val = add_features(X_val)
    X_te  = add_features(X_te)

    # Выравниваем колонки после feature engineering
    feat_cols = [c for c in X_tr.columns if c in X_val.columns and c in X_te.columns]
    X_tr, X_val, X_te = X_tr[feat_cols], X_val[feat_cols], X_te[feat_cols]

    numeric_cols = X_tr.select_dtypes(include=['object']).columns.tolist()
    if numeric_cols:
        print("Проблемные object-колонки:", numeric_cols)
        for col in numeric_cols:
            X_tr[col] = pd.to_numeric(X_tr[col], errors='coerce')
            X_val[col] = pd.to_numeric(X_val[col], errors='coerce')
            X_te[col] = pd.to_numeric(X_te[col], errors='coerce')

    # 6) Обучение LightGBM с early stopping
    model = LGBMRegressor(**LGB_PARAMS)
    model.fit(
        X_tr, y_tr,
        sample_weight      = w_tr,
        eval_set           = [(X_val, y_val)],
        eval_sample_weight = [w_val],
        eval_metric        = lgb_wmae_eval,
        callbacks          = [
            early_stopping(300, verbose=False),
            log_evaluation(500)
        ]
    )

    lgb_best_iters.append(model.best_iteration_)

    val_pred = np.clip(model.predict(X_val), 0, None)
    score = wmae(y_val, val_pred, w_val)
    fold_scores_lgb.append(score)

    oof_lgb[val_idx] = val_pred
    test_lgb += np.clip(model.predict(X_te), 0, None) / N_FOLDS

    print(f'  WMAE = {score:,.0f}  (best iter: {model.best_iteration_})')

oof_score_lgb = wmae(target_arr, oof_lgb, weights_arr)
print(f'\n==== LightGBM OOF WMAE: {oof_score_lgb:,.0f} ====')
print(f'Фолды: {[f"{s:,.0f}" for s in fold_scores_lgb]}')
print(f'Среднее лучшее число деревьев: {np.mean(lgb_best_iters):.0f}')


── Fold 1/5 ──
[500]	valid_0's l1: 65148.8	valid_0's wmae: 37010.7
[1000]	valid_0's l1: 64205.8	valid_0's wmae: 36692
  WMAE = 64,199  (best iter: 1188)

── Fold 2/5 ──
[500]	valid_0's l1: 60450.3	valid_0's wmae: 35969.8
[1000]	valid_0's l1: 60119.7	valid_0's wmae: 35808.9
  WMAE = 60,229  (best iter: 765)

── Fold 3/5 ──
[500]	valid_0's l1: 62297.3	valid_0's wmae: 36454.2
[1000]	valid_0's l1: 61926.3	valid_0's wmae: 36313.9
[1500]	valid_0's l1: 61758.8	valid_0's wmae: 36132
  WMAE = 61,766  (best iter: 1442)

── Fold 4/5 ──
[500]	valid_0's l1: 59688.1	valid_0's wmae: 36182.7
[1000]	valid_0's l1: 59111.2	valid_0's wmae: 36017.4
  WMAE = 59,011  (best iter: 1177)

── Fold 5/5 ──
[500]	valid_0's l1: 63552.3	valid_0's wmae: 36831.6
[1000]	valid_0's l1: 63160.2	valid_0's wmae: 36650.7
  WMAE = 63,102  (best iter: 1115)

==== LightGBM OOF WMAE: 61,681 ====
Фолды: ['64,199', '60,229', '61,766', '59,011', '63,102']
Среднее лучшее число деревьев: 1137


## 4. OOF Кросс-валидация — XGBoost

In [ ]:
# Параметры XGBoost
XGB_PARAMS = dict(
    objective         = 'reg:absoluteerror',
    n_estimators      = 10000,
    learning_rate     = 0.02,
    max_depth         = 7,
    min_child_weight  = 15,
    subsample         = 0.8,
    colsample_bytree  = 0.7,
    reg_alpha         = 0.1,
    reg_lambda        = 1.5,
    gamma             = 0.1,
    random_state      = SEED,
    n_jobs            = -1,
    tree_method       = 'hist',
    enable_categorical= False,
    early_stopping_rounds = 300,
)

oof_xgb  = np.zeros(len(train_raw))
test_xgb = np.zeros(len(test_raw))
fold_scores_xgb = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(train_raw)):
    print(f'\n── Fold {fold+1}/{N_FOLDS} ──')

    train_fold = train_raw.iloc[tr_idx].copy()
    valid_fold = train_raw.iloc[val_idx].copy()

    y_tr  = target_arr[tr_idx]
    y_val = target_arr[val_idx]
    w_tr  = weights_arr[tr_idx]
    w_val = weights_arr[val_idx]

    freq_maps = fit_frequency_encoding(train_fold)
    train_fold = apply_frequency_encoding(train_fold, freq_maps)
    valid_fold = apply_frequency_encoding(valid_fold, freq_maps)
    test_fold  = apply_frequency_encoding(test_raw.copy(), freq_maps)

    train_fold['__target__'] = y_tr
    te_maps, global_mean = fit_target_encoding(train_fold, y_tr)
    train_fold = train_fold.drop(columns=['__target__'])
    train_fold = apply_target_encoding(train_fold, te_maps, global_mean)
    valid_fold = apply_target_encoding(valid_fold, te_maps, global_mean)
    test_fold  = apply_target_encoding(test_fold,  te_maps, global_mean)

    drop_present = [c for c in DROP_COLS if c in train_fold.columns]
    X_tr  = train_fold.drop(columns=drop_present)
    X_val = valid_fold.drop(columns=[c for c in DROP_COLS if c in valid_fold.columns])
    X_te  = test_fold.drop(columns=[c for c in DROP_COLS if c in test_fold.columns])

    medians = X_tr.median()
    X_tr  = X_tr.fillna(medians)
    X_val = X_val.fillna(medians)
    X_te  = X_te.fillna(medians)

    common_cols = [c for c in X_tr.columns if c in X_val.columns and c in X_te.columns]
    X_tr, X_val, X_te = X_tr[common_cols], X_val[common_cols], X_te[common_cols]

    X_tr  = add_features(X_tr)
    X_val = add_features(X_val)
    X_te  = add_features(X_te)

    feat_cols = [c for c in X_tr.columns if c in X_val.columns and c in X_te.columns]
    X_tr, X_val, X_te = X_tr[feat_cols], X_val[feat_cols], X_te[feat_cols]

    model_xgb = XGBRegressor(**XGB_PARAMS)
    model_xgb.fit(
        X_tr, y_tr,
        sample_weight          = w_tr,
        eval_set               = [(X_val, y_val)],
        sample_weight_eval_set = [w_val],
        verbose                = 500,
    )

    val_pred = np.clip(model_xgb.predict(X_val), 0, None)
    score = wmae(y_val, val_pred, w_val)
    fold_scores_xgb.append(score)

    oof_xgb[val_idx] = val_pred
    test_xgb += np.clip(model_xgb.predict(X_te), 0, None) / N_FOLDS

    print(f'  WMAE = {score:,.0f}  (best iter: {model_xgb.best_iteration})')

oof_score_xgb = wmae(target_arr, oof_xgb, weights_arr)
print(f'\n==== XGBoost OOF WMAE: {oof_score_xgb:,.0f} ====')
print(f'Фолды: {[f"{s:,.0f}" for s in fold_scores_xgb]}')


── Fold 1/5 ──
[0]	validation_0-mae:133678.32984
[500]	validation_0-mae:66682.76323
[1000]	validation_0-mae:65889.68793
[1500]	validation_0-mae:65702.17987
[2000]	validation_0-mae:65578.62512
[2500]	validation_0-mae:65434.08899
[3000]	validation_0-mae:65324.58261
[3500]	validation_0-mae:65227.28615
[4000]	validation_0-mae:65181.77009
[4500]	validation_0-mae:65138.44139
[5000]	validation_0-mae:65083.90324
[5297]	validation_0-mae:65087.92655
  WMAE = 65,083  (best iter: 4997)

── Fold 2/5 ──
[0]	validation_0-mae:125108.05626
[500]	validation_0-mae:62057.11997
[1000]	validation_0-mae:61618.43243
[1500]	validation_0-mae:61521.76527
[2000]	validation_0-mae:61444.10704
[2500]	validation_0-mae:61403.63790
[2688]	validation_0-mae:61426.97108
  WMAE = 61,400  (best iter: 2388)

── Fold 3/5 ──
[0]	validation_0-mae:134422.91647
[500]	validation_0-mae:64393.27055
[1000]	validation_0-mae:63939.77331
[1500]	validation_0-mae:63773.79547
[1989]	validation_0-mae:63719.53647
  WMAE = 63,714  (best iter

## 5. Оптимальный ансамбль

In [ ]:
from scipy.optimize import minimize_scalar

# Находим оптимальный вес через перебор по OOF
def blend_wmae(alpha):
    blended = alpha * oof_lgb + (1 - alpha) * oof_xgb
    return wmae(target_arr, blended, weights_arr)

# Грубый перебор
alphas = np.linspace(0, 1, 101)
scores = [blend_wmae(a) for a in alphas]
best_alpha = alphas[np.argmin(scores)]

print(f'LightGBM OOF WMAE: {oof_score_lgb:,.0f}')
print(f'XGBoost  OOF WMAE: {oof_score_xgb:,.0f}')
print(f'Оптимальный вес LGB: {best_alpha:.2f}, XGB: {1-best_alpha:.2f}')
print(f'Ансамбль OOF WMAE:   {blend_wmae(best_alpha):,.0f}')

# Финальные предсказания
test_blend = np.clip(best_alpha * test_lgb + (1 - best_alpha) * test_xgb, 0, None)

LightGBM OOF WMAE: 61,681
XGBoost  OOF WMAE: 63,106
Оптимальный вес LGB: 0.95, XGB: 0.05
Ансамбль OOF WMAE:   61,677


## 6. Формирование submit

In [ ]:
commit_df = pd.DataFrame({
    'id':      test_ids,
    'predict': test_blend
})

commit_df.to_csv('commit.csv', sep=';', decimal=',', index=False)
print('Сохранено commit.csv')
print(commit_df['predict'].describe())
commit_df.head()

Сохранено commit.csv
count    7.321400e+04
mean     9.574341e+04
std      9.102937e+04
min      1.423646e+04
25%      4.171275e+04
50%      6.323797e+04
75%      1.169811e+05
max      1.107616e+06
Name: predict, dtype: float64


,id,predict
0,0,70968.896815
1,1,50498.355022
2,3,29239.548013
3,9,83061.900834
4,11,44383.569280


## 7. Попытка в CatBoost


In [ ]:
!pip install catboost

In [ ]:
from catboost import CatBoostRegressor, Pool

CAT_COLS = ['gender', 'adminarea', 'city_smart_name']  # нативная обработка

# Параметры CatBoost
CB_PARAMS = dict(
    loss_function    = 'MAE',
    iterations       = 10000,
    learning_rate    = 0.02,
    depth            = 8,
    l2_leaf_reg      = 3,
    random_seed      = SEED,
    verbose          = 500,
    early_stopping_rounds = 300,
)

oof_cb  = np.zeros(len(train_raw))
test_cb = np.zeros(len(test_raw))
fold_scores_cb = []

# Препроцессинг без кодирования категориальных
def preprocess_for_catboost(df, medians=None):
    df = df.copy()
    for col in df.columns:
        if col in DROP_COLS or col in CAT_COLS:
            continue
        if df[col].dtype == object:
            converted = pd.to_numeric(
                df[col].astype(str).str.replace(',', '.', regex=False), errors='coerce')
            if converted.notna().mean() > 0.5:
                df[col] = converted
            else:
                df = df.drop(columns=[col])
    for col in CAT_COLS:
        if col in df.columns:
            df[col] = df[col].fillna('MISSING').astype(str)
    num_cols = df.select_dtypes(include=[np.number]).columns
    num_cols = [c for c in num_cols if c not in DROP_COLS]
    if medians is None:
        medians = df[num_cols].median()
    df[num_cols] = df[num_cols].fillna(medians)
    return df, medians

for fold, (tr_idx, val_idx) in enumerate(kf.split(train_raw)):
    print(f'CatBoost Fold {fold+1}/{N_FOLDS}')

    tr_fold_cb, medians_cb = preprocess_for_catboost(train_raw.iloc[tr_idx].copy())
    val_fold_cb, _         = preprocess_for_catboost(train_raw.iloc[val_idx].copy(), medians_cb)
    te_fold_cb, _          = preprocess_for_catboost(test_raw.copy(), medians_cb)

    drop_present = [c for c in DROP_COLS if c in tr_fold_cb.columns]
    X_tr_cb  = tr_fold_cb.drop(columns=drop_present)
    X_val_cb = val_fold_cb.drop(columns=[c for c in DROP_COLS if c in val_fold_cb.columns])
    X_te_cb  = te_fold_cb.drop(columns=[c for c in DROP_COLS if c in te_fold_cb.columns])

    common = [c for c in X_tr_cb.columns if c in X_val_cb.columns and c in X_te_cb.columns]
    X_tr_cb, X_val_cb, X_te_cb = X_tr_cb[common], X_val_cb[common], X_te_cb[common]

    cat_idx = [i for i, c in enumerate(common) if c in CAT_COLS]

    y_tr  = target_arr[tr_idx]
    y_val = target_arr[val_idx]
    w_tr  = weights_arr[tr_idx]
    w_val = weights_arr[val_idx]

    tr_pool  = Pool(X_tr_cb,  y_tr,  weight=w_tr,  cat_features=cat_idx)
    val_pool = Pool(X_val_cb, y_val, weight=w_val, cat_features=cat_idx)

    cb_model = CatBoostRegressor(**CB_PARAMS)
    cb_model.fit(tr_pool, eval_set=val_pool)

    val_pred = np.clip(cb_model.predict(X_val_cb), 0, None)
    score = wmae(y_val, val_pred, w_val)
    fold_scores_cb.append(score)
    oof_cb[val_idx] = val_pred
    test_cb += np.clip(cb_model.predict(X_te_cb), 0, None) / N_FOLDS
    print(f'  WMAE = {score:,.0f}')

oof_score_cb = wmae(target_arr, oof_cb, weights_arr)
print(f'CatBoost OOF WMAE: {oof_score_cb:,.0f}')

# Трёхмодельный ансамбль с оптимальными весами
from scipy.optimize import minimize

def three_blend_wmae(weights):
    w = np.array(weights)
    w = w / w.sum()
    blended = w[0]*oof_lgb + w[1]*oof_xgb + w[2]*oof_cb
    return wmae(target_arr, blended, weights_arr)

res = minimize(three_blend_wmae, x0=[1/3, 1/3, 1/3],
               method='Nelder-Mead', options={'maxiter': 1000})
best_w = np.array(res.x) / np.sum(res.x)
print(f'Оптимальные веса: LGB={best_w[0]:.3f}, XGB={best_w[1]:.3f}, CB={best_w[2]:.3f}')

test_blend_3 = np.clip(best_w[0]*test_lgb + best_w[1]*test_xgb + best_w[2]*test_cb, 0, None)
commit_df['predict'] = test_blend_3
commit_df.to_csv('commit.csv', sep=';', decimal=',', index=False)
print(f'3-model ensemble OOF WMAE: {three_blend_wmae(best_w):,.0f}')

## 8. Анализ важности признаков

In [ ]:
# Важность признаков последнего LightGBM
try:
    importance = pd.DataFrame({
        'feature': feat_cols,
        'importance': model.feature_importances_
    }).sort_values('importance', ascending=False)

    print('Топ-30 важных признаков:')
    print(importance.head(30).to_string())

    print(f'\nПризнаков с нулевой важностью: {(importance.importance == 0).sum()}')
    zero_imp = importance[importance.importance == 0]['feature'].tolist()
    print('Кандидаты на удаление:', zero_imp[:20])
except NameError:
    print('Запустите сначала блок обучения LightGBM')